# Task 3 — CLIP Tasks

ATML PA0 — Task 3

Sections:
1. Zero-shot classification on STL-10 (3 prompting strategies)
2. Exploring the modality gap
3. Bridging the modality gap (Procrustes alignment)

## Setup

In [ ]:
# !pip install -q git+https://github.com/openai/CLIP.git ftfy regex scikit-learn scipy matplotlib

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import clip
from torchvision import datasets
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Available CLIP models:", clip.available_models())

os.makedirs("results", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

In [ ]:
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()
print("Loaded CLIP ViT-B/32")

## Data — STL-10

In [ ]:
# NOTE: use CLIP's own `preprocess` transform (handles CLIP-specific resize/normalize),
# NOT the ImageNet transforms used in Task 1/2.
stl_test = datasets.STL10(root="../data", split="test", download=True, transform=preprocess)

stl_classes = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck",
]
print(len(stl_test), "test images,", len(stl_classes), "classes")

## 1. Zero-Shot Classification on STL-10

In [ ]:
# Three prompting strategies
prompt_templates = {
    "plain": lambda c: c,
    "photo_of": lambda c: f"a photo of a {c}",
    "descriptive": lambda c: f"a high-quality photo of a {c}, a common object seen in everyday life",
}

In [ ]:
@torch.no_grad()
def build_text_features(template_fn, classes):
    texts = [template_fn(c) for c in classes]
    tokens = clip.tokenize(texts).to(device)
    text_features = model.encode_text(tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


@torch.no_grad()
def evaluate_zero_shot(loader, text_features):
    correct, total = 0, 0
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)
        image_features = model.encode_image(images)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarity = image_features @ text_features.T  # (batch, num_classes)
        preds = similarity.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total


from torch.utils.data import DataLoader
stl_loader = DataLoader(stl_test, batch_size=128, shuffle=False, num_workers=0)

zero_shot_results = {}
for name, template_fn in prompt_templates.items():
    text_feats = build_text_features(template_fn, stl_classes)
    acc = evaluate_zero_shot(stl_loader, text_feats)
    zero_shot_results[name] = acc
    print(f"{name}: {acc:.4f}")

In [ ]:
plt.figure(figsize=(5,4))
names = list(zero_shot_results.keys())
accs = list(zero_shot_results.values())
plt.bar(names, accs, color=["steelblue", "darkorange", "seagreen"])
plt.ylabel("zero-shot accuracy")
plt.title("CLIP zero-shot accuracy by prompting strategy (STL-10)")
for i, a in enumerate(accs):
    plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
plt.savefig("../figures/task3_zero_shot_prompting.png", dpi=150, bbox_inches="tight")
plt.show()

**Discussion:** Compare accuracy across the three prompting strategies. Why might more descriptive/templated prompts help?

> TODO: your answer here.

## 2. Exploring the Modality Gap

In [ ]:
N_SAMPLES = 100

from torch.utils.data import Subset
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(stl_test), size=N_SAMPLES, replace=False)
sample_subset = Subset(stl_test, sample_idx)
sample_loader = DataLoader(sample_subset, batch_size=32, shuffle=False, num_workers=0)

@torch.no_grad()
def extract_image_embeddings(loader):
    feats, labels_all = [], []
    for images, labels in loader:
        images = images.to(device)
        f = model.encode_image(images)
        f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu().numpy())
        labels_all.append(labels.numpy())
    return np.concatenate(feats), np.concatenate(labels_all)

image_embeds, sample_labels = extract_image_embeddings(sample_loader)

# Corresponding text embeddings: use each image's true class label as its "caption"
text_feats_photo = build_text_features(prompt_templates["photo_of"], stl_classes)
text_embeds = text_feats_photo.cpu().numpy()[sample_labels]  # one text embedding per image, matched by class

print("image_embeds:", image_embeds.shape, "text_embeds:", text_embeds.shape)

In [ ]:
from sklearn.manifold import TSNE

combined = np.concatenate([image_embeds, text_embeds], axis=0)
modality = np.array(["image"]*len(image_embeds) + ["text"]*len(text_embeds))

emb2d = TSNE(n_components=2, init="pca", random_state=0, perplexity=30).fit_transform(combined)

plt.figure(figsize=(6,6))
for m, color in zip(["image", "text"], ["steelblue", "darkorange"]):
    mask = modality == m
    plt.scatter(emb2d[mask, 0], emb2d[mask, 1], label=m, alpha=0.6, s=20, color=color)
plt.legend()
plt.title("t-SNE of CLIP image vs text embeddings (modality gap)")
plt.savefig("../figures/task3_modality_gap_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Quantify the gap: average cosine similarity within-modality vs across-modality,
# and the distance between the two modality centroids.
img_centroid = image_embeds.mean(axis=0)
txt_centroid = text_embeds.mean(axis=0)
centroid_gap = np.linalg.norm(img_centroid - txt_centroid)

matched_cos_sim = (image_embeds * text_embeds).sum(axis=1).mean()  # matched image-text pairs

print(f"Centroid distance (image vs text): {centroid_gap:.4f}")
print(f"Mean cosine similarity of matched image-text pairs: {matched_cos_sim:.4f}")

**Discussion:**
- How separated are the two modalities?
- Does normalization affect the modality gap?
- Why does CLIP still perform well despite this gap?

> TODO: your answer here.

## 3. Bridging the Modality Gap (Procrustes Alignment)

In [ ]:
from scipy.linalg import orthogonal_procrustes

# X = image embeddings, Y = text embeddings (matched pairs from Section 2)
X = image_embeds  # (N, 512)
Y = text_embeds    # (N, 512)

R, scale = orthogonal_procrustes(X, Y)
X_aligned = X @ R

print("Rotation matrix R shape:", R.shape)
print("Frobenius norm before alignment:", np.linalg.norm(X - Y))
print("Frobenius norm after alignment:", np.linalg.norm(X_aligned - Y))

In [ ]:
combined_aligned = np.concatenate([X_aligned, Y], axis=0)
emb2d_aligned = TSNE(n_components=2, init="pca", random_state=0, perplexity=30).fit_transform(combined_aligned)

plt.figure(figsize=(6,6))
for m, color in zip(["image", "text"], ["steelblue", "darkorange"]):
    mask = modality == m
    plt.scatter(emb2d_aligned[mask, 0], emb2d_aligned[mask, 1], label=m, alpha=0.6, s=20, color=color)
plt.legend()
plt.title("t-SNE after Procrustes alignment")
plt.savefig("../figures/task3_modality_gap_aligned_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Recompute zero-shot classification accuracy using the ALIGNED image embeddings
# against the (unaligned) text embeddings for all classes.

@torch.no_grad()
def evaluate_zero_shot_aligned(loader, text_features_np, R):
    correct, total = 0, 0
    text_features = torch.tensor(text_features_np, device=device, dtype=torch.float32)
    R_t = torch.tensor(R, device=device, dtype=torch.float32)
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)
        image_features = model.encode_image(images).float()
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        image_features_aligned = image_features @ R_t

        similarity = image_features_aligned @ text_features.T
        preds = similarity.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

text_feats_photo_np = text_feats_photo.cpu().numpy()
acc_before = zero_shot_results["photo_of"]
acc_after = evaluate_zero_shot_aligned(stl_loader, text_feats_photo_np, R)

print(f"Zero-shot accuracy BEFORE alignment: {acc_before:.4f}")
print(f"Zero-shot accuracy AFTER alignment:  {acc_after:.4f}")

**Discussion:** How does the Procrustes alignment affect the modality gap, and did it improve or hurt zero-shot accuracy? Why might that be?

> TODO: your answer here.

**Note:** the rotation R was fit using a 100-image subset. If accuracy drops after alignment, this is likely because R was learned on very little data and does not generalize to the full test set -- worth discussing as a limitation.